# Data Processing Pipeline with PySpark and Hadoop

This notebook demonstrates a realistic data processing pipeline using PySpark on Hadoop HDFS.

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

In [ ]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("Data-Processing-Pipeline") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-master:9000") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

print("Spark Session created!")
print(f"Spark Version: {spark.version}")

In [ ]:
# Step 1: Generate Sample Data
print("\n=== Step 1: Generate Sample Data ===")

# Create sample sales data
np.random.seed(42)
random.seed(42)

base_date = datetime(2024, 1, 1)
products = ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Headphones']
regions = ['US', 'EU', 'APAC', 'LATAM', 'EMEA']

# Generate 1000 sample records
sales_data = []
for i in range(1000):
    date = base_date + timedelta(days=random.randint(0, 90))
    product = random.choice(products)
    region = random.choice(regions)
    quantity = random.randint(1, 100)
    price = np.random.uniform(10, 1000)
    
    sales_data.append({
        'date': date.strftime('%Y-%m-%d'),
        'product': product,
        'region': region,
        'quantity': quantity,
        'unit_price': round(price, 2),
        'total_sales': round(quantity * price, 2)
    })

print(f"Generated {len(sales_data)} sales records")
print(f"Sample records: {sales_data[:3]}")

In [ ]:
# Step 2: Create DataFrame from sample data
print("\n=== Step 2: Create DataFrame ===")

schema = StructType([
    StructField("date", StringType(), True),
    StructField("product", StringType(), True),
    StructField("region", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("total_sales", DoubleType(), True)
])

df_sales = spark.createDataFrame(sales_data, schema=schema)

print(f"DataFrame created with {df_sales.count()} rows")
print("\nSchema:")
df_sales.printSchema()

print("\nFirst 10 rows:")
df_sales.show(10)

In [ ]:
# Step 3: Save raw data to HDFS
print("\n=== Step 3: Save Raw Data to HDFS ===")

raw_data_path = "hdfs://hadoop-master:9000/user/jupyter/sales_raw"

try:
    df_sales.write.mode("overwrite").csv(raw_data_path, header=True)
    print(f"Raw data saved to: {raw_data_path}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Step 4: Data Cleaning and Transformation
print("\n=== Step 4: Data Cleaning and Transformation ===")

# Convert date to proper type
df_clean = df_sales \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("day_of_week", dayofweek(col("date"))) \
    .filter(col("quantity") > 0) \
    .filter(col("unit_price") > 0)

print(f"After cleaning: {df_clean.count()} rows")
print("\nData sample:")
df_clean.show(5)

In [ ]:
# Step 5: Aggregation and Analysis
print("\n=== Step 5: Aggregation and Analysis ===")

# Total sales by product
print("\n1. Total Sales by Product:")
sales_by_product = df_clean.groupBy("product") \
    .agg(
        sum(col("total_sales")).alias("total_revenue"),
        sum(col("quantity")).alias("total_quantity"),
        count("*").alias("transaction_count"),
        avg(col("unit_price")).alias("avg_price")
    ) \
    .sort(col("total_revenue").desc())

sales_by_product.show()

In [ ]:
# Total sales by region
print("\n2. Total Sales by Region:")
sales_by_region = df_clean.groupBy("region") \
    .agg(
        sum(col("total_sales")).alias("total_revenue"),
        avg(col("total_sales")).alias("avg_transaction"),
        count("*").alias("transaction_count")
    ) \
    .sort(col("total_revenue").desc())

sales_by_region.show()

In [ ]:
# Sales by product and region
print("\n3. Sales by Product and Region (Top 15):")
sales_by_product_region = df_clean.groupBy("product", "region") \
    .agg(
        sum(col("total_sales")).alias("revenue"),
        count("*").alias("count")
    ) \
    .sort(col("revenue").desc())

sales_by_product_region.limit(15).show()

In [ ]:
# Step 6: Window Functions (Advanced)
print("\n=== Step 6: Window Functions ===")

from pyspark.sql.window import Window

# Rank products by sales within each region
windowSpec = Window.partitionBy("region").orderBy(col("total_sales").desc())

df_ranked = df_clean.withColumn(
    "rank_in_region",
    row_number().over(windowSpec)
)

print("\nTop 3 products in each region:")
df_ranked.filter(col("rank_in_region") <= 3) \
    .select("region", "product", "total_sales", "rank_in_region") \
    .sort("region", "rank_in_region") \
    .show(15)

In [ ]:
# Step 7: Join Operations
print("\n=== Step 7: Join Operations ===")

# Create product dimension
product_dim_data = [
    ("Laptop", "Electronics", "High"),
    ("Mouse", "Electronics", "Low"),
    ("Keyboard", "Electronics", "Medium"),
    ("Monitor", "Electronics", "High"),
    ("Headphones", "Audio", "Medium")
]

product_schema = StructType([
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price_tier", StringType(), True)
])

df_product_dim = spark.createDataFrame(product_dim_data, schema=product_schema)

# Join with sales data
df_enriched = df_clean.join(df_product_dim, on="product", how="left")

print("\nEnriched data (with category):")
df_enriched.select("date", "product", "category", "region", "total_sales") \
    .show(10)

In [ ]:
# Step 8: Save processed data to HDFS
print("\n=== Step 8: Save Processed Data ===")

processed_path = "hdfs://hadoop-master:9000/user/jupyter/sales_processed"
aggregated_path = "hdfs://hadoop-master:9000/user/jupyter/sales_aggregated"
enriched_path = "hdfs://hadoop-master:9000/user/jupyter/sales_enriched"

try:
    df_clean.write.mode("overwrite").parquet(processed_path)
    print(f"Processed data saved to: {processed_path}")
    
    sales_by_product.write.mode("overwrite").parquet(aggregated_path)
    print(f"Aggregated data saved to: {aggregated_path}")
    
    df_enriched.write.mode("overwrite").parquet(enriched_path)
    print(f"Enriched data saved to: {enriched_path}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Step 9: Statistics and Summary
print("\n=== Step 9: Statistics and Summary ===")

print(f"\nTotal Records Processed: {df_clean.count()}")
print(f"Total Revenue: ${df_clean.agg(sum('total_sales')).collect()[0][0]:,.2f}")
print(f"Average Transaction Value: ${df_clean.agg(avg('total_sales')).collect()[0][0]:,.2f}")
print(f"Max Transaction: ${df_clean.agg(max('total_sales')).collect()[0][0]:,.2f}")
print(f"Min Transaction: ${df_clean.agg(min('total_sales')).collect()[0][0]:,.2f}")

df_clean.describe().show()

In [ ]:
# Step 10: Visualization
print("\n=== Step 10: Visualization ===")

import matplotlib.pyplot as plt
import pandas as pd

# Convert to Pandas for visualization
sales_by_product_pd = sales_by_product.toPandas()
sales_by_region_pd = sales_by_region.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Sales by Product
axes[0].bar(sales_by_product_pd['product'], sales_by_product_pd['total_revenue'], color='steelblue')
axes[0].set_title('Total Revenue by Product')
axes[0].set_ylabel('Revenue ($)')
axes[0].set_xlabel('Product')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Sales by Region
axes[1].bar(sales_by_region_pd['region'], sales_by_region_pd['total_revenue'], color='coral')
axes[1].set_title('Total Revenue by Region')
axes[1].set_ylabel('Revenue ($)')
axes[1].set_xlabel('Region')

plt.tight_layout()
plt.show()

print("Charts displayed!")

In [ ]:
# Step 11: SQL Queries
print("\n=== Step 11: SQL Queries ===")

# Create temporary view
df_enriched.createOrReplaceTempView("sales")

# Complex SQL query
result = spark.sql("""
    SELECT 
        product,
        category,
        region,
        SUM(total_sales) as revenue,
        COUNT(*) as transactions,
        AVG(total_sales) as avg_transaction,
        SUM(quantity) as total_quantity
    FROM sales
    GROUP BY product, category, region
    HAVING SUM(total_sales) > 5000
    ORDER BY revenue DESC
""")

print("\nHigh-value sales combinations (revenue > $5000):")
result.show(20)

In [ ]:
# Step 12: Performance metrics
print("\n=== Step 12: Performance Metrics ===")

sc = spark.sparkContext
print(f"\nSpark Context:")
print(f"  Master: {sc.master}")
print(f"  App Name: {sc.appName}")
print(f"  Default Parallelism: {sc.defaultParallelism}")
print(f"  Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

print(f"\nDataFrame Info:")
print(f"  Partitions: {df_clean.rdd.getNumPartitions()}")
print(f"  Row Count: {df_clean.count()}")

In [ ]:
# Stop Spark Session
print("\nCleaning up...")
spark.stop()
print("Data processing pipeline completed!")